In [ ]:
# %% [markdown]
# # 📊 Posições com Cotação de Mercado
#
# Valor de mercado em tempo real, lucro/prejuízo e rentabilidade.

# %%
import sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if ROOT.name == "valuation":
    ROOT = ROOT.parent.parent
elif ROOT.name in ("src", "notebooks"):
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env")
sys.path.insert(0, str(ROOT))

from src.collectors.supabase_client import load_transactions
from src.portfolio.positions import calculate_positions, get_open_positions
from src.portfolio.market_data import enrich_with_market_data, portfolio_totals
from src.reports.accessible import market_summary

In [ ]:
# %% [markdown]
# ## 1. Carregar transações e calcular posições

# %%
df = load_transactions()
posicoes = calculate_positions(df)
abertas = get_open_positions(posicoes)
print(f"✅ {len(abertas)} posições em aberto.")

In [ ]:
# %% [markdown]
# ## 2. Buscar cotações e enriquecer

# %%
print("⏳ Buscando cotações no Yahoo Finance...")
enriched = enrich_with_market_data(abertas)
totals = portfolio_totals(enriched)
print("✅ Cotações carregadas!")

In [ ]:
# %% [markdown]
# ## 3. Resumo completo com mercado

# %%
print(market_summary(enriched, totals))

In [ ]:
# %% [markdown]
# ## 4. Tabela completa (DataFrame)

# %%
enriched[[
    "ticker", "nome", "categoria", "moeda",
    "qtde_saldo", "preco_medio_brl", "preco_atual_brl",
    "custo_total_brl", "valor_mercado_brl",
    "lucro_prejuizo_brl", "rentabilidade_pct",
]].sort_values("valor_mercado_brl", ascending=False)

In [ ]:
# %% [markdown]
# ## 5. Maiores ganhos e perdas

# %%
print("=" * 60)
print("🟢 MAIORES GANHOS")
print("=" * 60)
gains = enriched[enriched["lucro_prejuizo_brl"] > 0].nlargest(5, "lucro_prejuizo_brl")
for _, r in gains.iterrows():
    rent = f"{r['rentabilidade_pct']:+.1f}%"
    print(f"  📈 {r['ticker']:12s} | L/P: {r['lucro_prejuizo_brl']:>12,.2f} BRL | Rent: {rent}")

print()
print("=" * 60)
print("🔴 MAIORES PERDAS")
print("=" * 60)
losses = enriched[enriched["lucro_prejuizo_brl"] < 0].nsmallest(5, "lucro_prejuizo_brl")
for _, r in losses.iterrows():
    rent = f"{r['rentabilidade_pct']:+.1f}%"
    print(f"  📉 {r['ticker']:12s} | L/P: {r['lucro_prejuizo_brl']:>12,.2f} BRL | Rent: {rent}")
